# Filter the 10,000 queries (automatic)
Replaces manual checking with rules. Every removed row gets a **reason**, so you can spot-check a sample instead of reading all 10,000.
Input: `data/processed/dataset_10000_queries.csv`. Output: `dataset_10000_filtered.csv` (kept) and `dataset_10000_removed.csv` (removed + reason).

In [ ]:
import re, csv, collections, random, unicodedata
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
IN = ROOT / "data/processed/dataset_10000_queries.csv"
rows = list(csv.DictReader(open(IN, encoding="utf-8-sig")))
print(len(rows), "rows")

## 1. Rules
Each rule returns a reason string or `None`. Change the lists to tune.

In [ ]:
PRAISE = re.compile(r"ধন্যবাদ|থ্যাংক|জাযাকাল|জাজাকাল|আল্লাহ|আলহামদুলিল|ভালো লাগ|সুন্দর ভিডিও|দোয়া|আমিন|মাশা\s?আল্লাহ|thank|great video|nice video|love from|subscribe|helpful|waalaikum", re.I)
AD_LINK = re.compile(r"https?://|www\.|\.com\b|\bwhatsapp\b|whats app|\b01\d{9}\b|\+?880\d{9,10}|telegram|inbox me|যোগাযোগ করুন|ইনবক্স", re.I)
E = r"(?![ঀ-৿])"   # Bengali-safe word end (\b fails after vowel signs)
Q_BN = re.compile(r"কি" + E + "|কী" + E + "|কত" + E + "|কেন" + E + "|কিভাবে|কীভাবে|কখন|কোথায়|কোথা|কোন" + E + "|কোনটি|কে" + E + "|কাদের|কার" + E + "|কতদিন|কয়|নিয়ম|পদ্ধতি|উপায়|সম্ভব|যাবে|যায়|পারবো|পারব|পারি|করবো|করব|করতে|করা|হবে|হয়|জানতে|জানাবেন|বলবেন|বলুন|লাগে|লাগবে|আছে কি|কেনো|কেমনে|বলেন|বলো|দিন|দেন|জানান|কিছু|\\?|？")
Q_EN = re.compile(r"\b(what|how|why|when|where|which|who|can|could|is|are|do|does|will|should|kivabe|kemon|kothay|koto|kobe|keno|ki|kon|ase|ache|jabe|hobe|korbo|korte|lagbe|jante|janaben|bolben)\b|\?", re.I)

def words(t): return re.findall(r"[ঀ-৿A-Za-z0-9]+", t)

def reason(t):
    t = t.strip()
    w = words(t)
    if len(w) < 4: return "too_short"
    if len(w) > 80: return "too_long"
    if AD_LINK.search(t): return "ad_or_link"
    if PRAISE.search(t) and not re.search(r"\?|কিভাবে|কীভাবে|কত" + E + "|কি" + E, t): return "praise_or_greeting"
    body = re.sub(r"[\s.,;:!?।\-_/\\'\"()\[\]]", "", t)
    letters = [c for c in body if c.isalnum() or "\u0980" <= c <= "\u09FF"]
    if body and len(letters) < 0.6 * len(body): return "mostly_symbols_or_emoji"
    if re.search(r"([\u0980-\u09FFA-Za-z])\1{5,}", t): return "repeated_chars"
    if not (Q_BN.search(t) or Q_EN.search(t)): return "no_question_signal"
    return None

## 2. Apply rules

In [ ]:
for r in rows:
    r["reason"] = reason(r["text"]) or ""
print(collections.Counter(r["reason"] or "KEEP" for r in rows).most_common())

## 3. Remove duplicates
Exact duplicates after normalisation, then near-duplicates (word-set overlap ≥ 0.8). The first occurrence is kept.

In [ ]:
def norm(t):
    t = unicodedata.normalize("NFKC", t).lower()
    return " ".join(words(t))

seen_exact, kept_sets = {}, []
for r in rows:
    if r["reason"]: continue
    n = norm(r["text"]); r["_n"] = n
    if n in seen_exact:
        r["reason"] = "duplicate"; continue
    s = set(n.split())
    dup = False
    for s2 in kept_sets[-3000:]:          # compare with recent kept rows (fast enough for 10k)
        if len(s & s2) / max(1, len(s | s2)) >= 0.8:
            dup = True; break
    if dup: r["reason"] = "near_duplicate"; continue
    seen_exact[n] = 1; kept_sets.append(s)
print(collections.Counter(r["reason"] or "KEEP" for r in rows).most_common())

## 4. Assign a domain to the Facebook rows (keywords)
Only fills empty domains. Rows with no keyword hit stay `unassigned`.

In [ ]:
KW = {
 "fees": r"ফি\b|ফী|খরচ|টাকা|চার্জ|ট্যাক্স|কর\b|বিল|ব্যাংক|বিকাশ|নগদ|সুদ|ঋণ|লোন|বেতন|ভ্যাট|fee|charge|cost|tax|bkash|nagad|loan|taka",
 "procedures": r"এনআইডি|জন্ম নিবন্ধন|পাসপোর্ট|লাইসেন্স|নামজারি|দলিল|খতিয়ান|আবেদন|নিবন্ধন|সনদ|সার্টিফিকেট|ভিসা|passport|nid|license|apply|registration",
 "health": r"জ্বর|ডেঙ্গু|ডায়াবেটিস|টিকা|ডাক্তার|হাসপাতাল|রোগ|ব্যথা|ওষুধ|চিকিৎসা|প্রেসার|গর্ভ|ক্যান্সার|doctor|fever|dengue|medicine|vaccine",
 "education": r"ভর্তি|পরীক্ষা|এসএসসি|এইচএসসি|বিশ্ববিদ্যালয়|স্কলারশিপ|বৃত্তি|বিসিএস|স্কুল|কলেজ|ফলাফল|ইংরেজি|admission|exam|scholarship|ielts|university",
 "tech": r"ফেসবুক|পাসওয়ার্ড|মোবাইল|ফোন|অ্যাপ|এ্যাপ|ইন্টারনেট|ওয়াইফাই|ল্যাপটপ|কম্পিউটার|ফ্রিল্যান্স|আইডি হ্যাক|facebook|password|wifi|app|laptop|freelanc",
 "local_services": r"ট্রেন|টিকিট|বাস|মেট্রো|বিদ্যুৎ|গ্যাস|পানি|৯৯৯|হেল্পলাইন|ঢাকা|ফ্লাইট|ঠিকানা|train|ticket|metro|bus|flight|helpline",
}
KW = {d: re.compile(p, re.I) for d, p in KW.items()}
filled = 0
for r in rows:
    if r["domain"]: continue
    hits = {d: len(p.findall(r["text"])) for d, p in KW.items()}
    d, n = max(hits.items(), key=lambda x: x[1])
    r["domain"] = d if n else "unassigned"; filled += 1
print("filled", filled, collections.Counter(r["domain"] for r in rows if r["source"] == "facebook"))

## 5. Save kept and removed

In [ ]:
kept = [r for r in rows if not r["reason"]]
removed = [r for r in rows if r["reason"]]
cols = ["id", "source", "domain", "script", "text"]
def save(path, data, extra=()):
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols + list(extra), extrasaction="ignore"); w.writeheader(); w.writerows(data)
save(ROOT / "data/processed/dataset_10000_filtered.csv", kept)
save(ROOT / "data/processed/dataset_10000_removed.csv", removed, ["reason"])
print("kept", len(kept), "removed", len(removed))
print(collections.Counter(r["script"] for r in kept), collections.Counter(r["domain"] for r in kept))

## 6. Spot-check (read ~20 of each instead of 10,000)

In [ ]:
random.seed(1)
print("--- KEPT sample ---")
for r in random.sample(kept, 20): print(r["domain"], "|", r["text"][:120].replace("\n", " "))
print("\n--- REMOVED sample ---")
for r in random.sample(removed, 20): print(r["reason"], "|", r["text"][:120].replace("\n", " "))